<a href="https://colab.research.google.com/github/Emmanuel2211/.config/blob/master/Entregable_Tarea_5_1_Alfaro_Cruz_P%C3%A9rez_Rupit_Medea.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Contaminación Atmosférica y Calidad del Aire (PM2.5)

## Exploración y Preprocesamiento
15% Manejo adecuado de valores nulos, codificación de categóricas y escalamiento de variables (crítico para GLM, L1/$L_2$ y Redes Neuronales).

## Resumen de la programación realizada en este código

1) ashdgjasgdjha
2) ajsdajhds


In [18]:
# Incluir librerías necesarias para procesamiento y graficación
import pandas as pd # Manejo de datos en tablas
import numpy as np # Operaciones numéricas
import matplotlib.pyplot as plt  # Gráficas
import seaborn as sns  # Gráficas más avanzadas

In [19]:
# Importar la información
df = pd.read_csv("sample_data/PRSA_data_2010.1.1-2014.12.31.csv")

In [20]:
# Mostrar las primeras 5 filas
df.sample(10)

,No,year,month,day,hour,pm2.5,DEWP,TEMP,PRES,cbwd,Iws,Is,Ir
16866,16867,2011,12,4,18,498.0,-3,-3.0,1029.0,SE,8.95,0,0
20654,20655,2012,5,10,14,165.0,15,29.0,1010.0,SE,8.05,0,0
422,423,2010,1,18,14,154.0,-9,1.0,1025.0,NE,1.79,0,0
7154,7155,2010,10,26,2,7.0,-13,3.0,1040.0,NW,4.92,0,0
27158,27159,2013,2,5,14,89.0,-3,-2.0,1022.0,SE,1.79,0,0
12662,12663,2011,6,12,14,22.0,9,33.0,1005.0,cv,3.13,0,0
13424,13425,2011,7,14,8,94.0,22,26.0,1006.0,SE,3.58,0,0
31422,31423,2013,8,2,6,63.0,19,21.0,1005.0,cv,0.45,0,0
1186,1187,2010,2,19,10,112.0,-15,1.0,1018.0,NE,4.47,0,0
39020,39021,2014,6,14,20,58.0,19,28.0,1005.0,SE,12.97,0,0


# Glosario de Columnas Relevantes

pm2.5 --> PM25: Concentración de partículas PM2.5 (material particulado fino)

DEWP: Punto de rocío (Dew Point)

TEMP: Temperatura ambiente

PRES: Presión atmosférica

cbwd --> WIND_DIRECTION : Dirección combinada del viento

lws --> WIND_SPEED: velocidad acumulada del viento

ls: Horas acumuladas de nieve

lr: Horas acumuladas de lluvia

 -NW: Viento del noroeste

 -NE: Viento del noreste

 -SE: Viento del sureste

 -cv: Calm Variable (viento muy debil o dirección variable). *Combinación de las tres anteriores, sin rumbo fijo.

In [21]:
# Revisar el tipos de datos y columnas presentes
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43824 entries, 0 to 43823
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   No      43824 non-null  int64  
 1   year    43824 non-null  int64  
 2   month   43824 non-null  int64  
 3   day     43824 non-null  int64  
 4   hour    43824 non-null  int64  
 5   pm2.5   41757 non-null  float64
 6   DEWP    43824 non-null  int64  
 7   TEMP    43824 non-null  float64
 8   PRES    43824 non-null  float64
 9   cbwd    43824 non-null  object 
 10  Iws     43824 non-null  float64
 11  Is      43824 non-null  int64  
 12  Ir      43824 non-null  int64  
dtypes: float64(4), int64(8), object(1)
memory usage: 4.3+ MB


In [22]:
#Renombrar encabezados
df = df.rename(columns={"No": "NO",
                        "year": "YEAR",
                        "month": "MONTH",
                        "day": "DAY",
                        "hour": "HOUR",
                        "pm2.5": "PM25",
                        "DEWP": "DEWPOINT",
                        "TEMP": "TEMP",
                        "PRES": "PRESION",
                        "cbwd": "WIND_DIRECTION",
                        "Iws": "WINSD_SPEED",
                        "Is": "SNOW",
                        "Ir": "RAIN"})
df.info()  # Revisar el tipos de datos y columnas presentes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43824 entries, 0 to 43823
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   NO              43824 non-null  int64  
 1   YEAR            43824 non-null  int64  
 2   MONTH           43824 non-null  int64  
 3   DAY             43824 non-null  int64  
 4   HOUR            43824 non-null  int64  
 5   PM25            41757 non-null  float64
 6   DEWPOINT        43824 non-null  int64  
 7   TEMP            43824 non-null  float64
 8   PRESION         43824 non-null  float64
 9   WIND_DIRECTION  43824 non-null  object 
 10  WINSD_SPEED     43824 non-null  float64
 11  SNOW            43824 non-null  int64  
 12  RAIN            43824 non-null  int64  
dtypes: float64(4), int64(8), object(1)
memory usage: 4.3+ MB


In [24]:
# Crear una columna con la fecha a partir de year, month, day y hour
# Mostrar resultados
df['FECHA'] = pd.to_datetime(df[['YEAR', 'MONTH', 'DAY', 'HOUR']]) # Crea una columna con la fecha
print("Rango de fechas del dataset:")
print(f"Desde: {df['FECHA'].min()} hasta {df['FECHA'].max()}")

Rango de fechas del dataset:
Desde: 2010-01-01 00:00:00 hasta 2014-12-31 23:00:00


In [25]:
# Análisis de métricas principales de la información
df.drop(columns=["NO", "YEAR", "MONTH", "DAY", "HOUR"]).describe()   # Estadísticas básicas sin considerar las fechas

,PM25,DEWPOINT,TEMP,PRESION,WINSD_SPEED,SNOW,RAIN,FECHA
count,41757.000000,43824.000000,43824.000000,43824.000000,43824.000000,43824.000000,43824.000000,43824
mean,98.613215,1.817246,12.448521,1016.447654,23.889140,0.052734,0.194916,2012-07-01 23:30:00
min,0.000000,-40.000000,-19.000000,991.000000,0.450000,0.000000,0.000000,2010-01-01 00:00:00
25%,29.000000,-10.000000,2.000000,1008.000000,1.790000,0.000000,0.000000,2011-04-02 11:45:00
50%,72.000000,2.000000,14.000000,1016.000000,5.370000,0.000000,0.000000,2012-07-01 23:30:00
75%,137.000000,15.000000,23.000000,1025.000000,21.910000,0.000000,0.000000,2013-10-01 11:15:00
max,994.000000,28.000000,42.000000,1046.000000,585.600000,27.000000,36.000000,2014-12-31 23:00:00
std,92.050387,14.433440,12.198613,10.268698,50.010635,0.760375,1.415867,NaN


Nota: El rango de las variables (diferente entre ellas) puede afectar el modelo por lo que es posible que se deba normalizar

In [68]:
threshold = df['PM25'].quantile(0.99)
filtered_df = df[df['PM25'] >= threshold]
filtered_df['PM25'].describe()

,PM25
count,423.000000
mean,493.101655
std,87.652354
min,420.000000
25%,439.000000
50%,470.000000
75%,508.000000
max,994.000000


En este bloque revisamos los outliers variando el percentil, no eliminamos dato.

In [28]:
# Revisión de datos nulos de la información
df.isnull().sum()

,0
NO,0
YEAR,0
MONTH,0
DAY,0
HOUR,0
PM25,2067
DEWPOINT,0
TEMP,0
PRESION,0
WIND_DIRECTION,0


In [36]:
# Porcentaje de vacíos en la información por modelar
x = 2067/43824*100
print(round(x, 3), '%')

4.717 %


Sólo la variable por modelar tiene valores nulos y es menos del 4.72%

In [ ]:
# Configuración de los gráficos
plt.figure(figsize=(12, 4))

# 1. Gráfico para ver la distribución de pm2.5 e identificar sesgos o valores atípicos (outliers)
plt.subplot(1, 2, 1)
sns.histplot(df['PM25'].dropna(), bins=50, kde=True, color='grey')
plt.title('Distribución de Partículas pm2.5')
plt.xlabel('Concentración μg/m³')

In [ ]:
# 2. Gráfico de caja para ver cómo afecta el viento en calma ('cv') frente a los otros a las PM2.5
plt.subplot(1, 2, 2)
sns.boxplot(x='cbwd', y='pm2.5', data=df)
plt.title('PM2.5 por Dirección del Viento')
plt.xlabel('Dirección (cbwd)')
plt.tight_layout()
plt.show()

In [ ]:
# Crear un índice con la fecha
df_temporal = df.set_index('fecha') # Hacer un índice con la fecha
# Graficar dos columnas en la misma serie temporal
plt.figure(figsize=(10,6))
plt.plot(df_temporal, df["pm2.5"], label='Concentración', color='orange')
plt.plot(df_temporal, df['DEWP'], label='Rocío', color='grey')

In [ ]:
plt.plot(df["fecha"], df["pm2.5"], label="PM2.5")
plt.plot(df["fecha"], df["DEWP"], label="DEWP")

In [ ]:
# Rellenar nulos con la mediana
# df['pm2.5'] = df['pm2.5'].fillna(df['pm2.5'].median())

Opción 1: Eliminar nulos

Opción 2: Rellenar con la media

Opción 3: Rellenar con interpolación temporal

In [69]:
df_cleaned = df.dropna(subset=['PM25'])
df_inputed = df['PM25'].fillna(df['PM25'].median(), inplace=True)
print(len(df_cleaned))
print(len(df))

41757
43824


/tmp/ipykernel_6018/1343685226.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_inputed = df['PM25'].fillna(df['PM25'].median(), inplace=True)


In [ ]:
g = sns.PairGrid(df_cleaned)
g.map(sns.scatterplot)